In [13]:
from google.cloud import storage
from dotenv import load_dotenv
import os
import pandas as pd
from io import BytesIO

load_dotenv()

True

In [14]:
GCS_BUCKET = os.getenv("GCS_BUCKET", "").strip()
PREFIX = os.getenv("GOLD_PREFIX", "").strip().strip("/")
SEASON = int(os.getenv("SEASON", ""))

GOLD_PATH = f"gold/season={SEASON}/round"

In [15]:
client = storage.Client()
bucket = client.bucket(GCS_BUCKET)

In [16]:
blobs = [
    b for b in client.list_blobs(GCS_BUCKET, prefix=GOLD_PATH)
    if b.name.endswith("report.parquet")
]

In [17]:
cols = ["Driver", "DriverNumber", "Team", "season", "Points"]

parts = []

for blob in blobs:
    print(blob.name)
    df = pd.read_parquet(BytesIO(blob.download_as_bytes()))
    parts.append(df[cols].copy())

all_rounds = pd.concat(parts, ignore_index=True)

all_rounds

gold/season=2026/round=01/report.parquet
gold/season=2026/round=02/report.parquet
gold/season=2026/round=03/report.parquet


,Driver,DriverNumber,Team,season,Points
0,RUS,63,Mercedes,2026,25
1,ANT,12,Mercedes,2026,18
2,LEC,16,Ferrari,2026,15
3,HAM,44,Ferrari,2026,12
4,NOR,1,McLaren,2026,10
...,...,...,...,...,...
61,ALO,14,Aston Martin,2026,0
62,BOT,77,Cadillac,2026,0
63,ALB,23,Williams,2026,0
64,STR,18,Aston Martin,2026,0


In [18]:
standings = (
    all_rounds.groupby("DriverNumber", as_index=False)
    .agg(
        Points = ("Points", "sum"),
        Driver = ("Driver", "first"),
        Team = ("Team", "first"),
        season = ("season", "first"),
    )
    .sort_values("Points", ascending=False)
    .reset_index(drop=True)
)

standings

stan_buf = BytesIO()
standings.to_parquet(stan_buf, index=False)
stan_buf.seek(0)
bucket.blob(f"{GOLD_PATH}/standings.parquet").upload_from_file(stan_buf)
standings.to_csv("./data/standings.csv")

In [19]:
team_standings = (
    standings.groupby("Team", as_index=False)
    .agg(
        Points = ("Points", "sum"),
        Season = ("season", "first"),
        
    )
    .sort_values("Points", ascending=False)
    .reset_index(drop=True)
)

team_buf = BytesIO()
team_standings.to_parquet(team_buf, index=False)
team_buf.seek(0)
bucket.blob(f"{GOLD_PATH}/team_standings.parquet").upload_from_file(team_buf)

team_standings.to_csv("./data/team_standings.csv", index=False)
